In [0]:
%run ./00_utils

In [0]:
# =============================================================================
# Notebook: 02_ingest_incremental
# Purpose: Connects to the source database, reads the Watermark, pulls ONLY 
#          new records, validates schema using modules, and writes to Bronze.
# =============================================================================

from pyspark.sql import functions as F

storage_account = "stsupplychaindata"
mock_source_path = f"abfss://system@{storage_account}.dfs.core.windows.net/mock_source_db/"
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/inventory_shipments/"
watermark_path = f"abfss://system@{storage_account}.dfs.core.windows.net/watermark_table/"
table_name = "inventory_shipments"

# ---- Step 1: Get the Last Watermark (Using Reusable Module) ----
print("--- 1. Fetching Watermark ---")
last_watermark = get_watermark(spark, watermark_path, table_name)

# ---- Step 2: Read Source & Filter for New Data ----
print(f"\n--- 2. Extracting data created AFTER {last_watermark} ---")
df_source = spark.read.format("delta").load(mock_source_path)

# INCREMENTAL FILTER LOGIC
df_new_data = df_source.filter(F.col("created_at") > last_watermark)
new_record_count = df_new_data.count()
print(f"Found {new_record_count} new records to ingest.")

if new_record_count > 0:
    # ---- Step 3: Validate Schema (Using Reusable Module) ----
    print("\n--- 3. Validating Schema ---")
    expected_schema = ["shipment_id", "warehouse_id", "product_sku", "quantity", "created_at"]
    validate_schema(df_new_data, expected_schema)
    
    # ---- Step 4: Write to Bronze (Append) ----
    print("\n--- 4. Writing to Bronze Layer ---")
    # We use "append" because we are only adding the NEW delta of records
    df_new_data.write.mode("append").format("delta").save(bronze_path)
    print("Data successfully ingested to Bronze.")
    
    # ---- Step 5: Update Watermark (Using Reusable Module) ----
    print("\n--- 5. Updating Watermark ---")
    new_high_watermark = df_new_data.agg(F.max("created_at")).collect()[0][0]
    update_watermark(spark, watermark_path, table_name, new_high_watermark)
    
else:
    print("\n No new data to process. Pipeline completed successfully.")

--- 1. Fetching Watermark ---
⚠ No watermark found. Defaulting to 1900-01-01.

--- 2. Extracting data created AFTER 1900-01-01 00:00:00 ---
Found 1000 new records to ingest.

--- 3. Validating Schema ---
✅ Reusable Module: Schema Validation Passed!

--- 4. Writing to Bronze Layer ---
Data successfully ingested to Bronze.

--- 5. Updating Watermark ---
Reusable Module: Watermark updated to -> 2024-01-31 22:00:00
